In [26]:
import xarray as xr
import gfdl_utils as gu
import glob
import cftime
import numpy as np
import esm4ppe
from dask.diagnostics import ProgressBar

# Load data

In [27]:
basedir = '/projects/SOCCOM/datasets/archive/kbr/fre/verona/ESM2M/20161113/ANALYSIS/'
variable = 'NPP'
if variable in ['SST','MLD']:
    filenamevariable = variable.lower()
    time = 'time'
else:
    filenamevariable = variable
    time='TIME'

## Load ensemble

In [28]:
filename = filenamevariable+'_ENS*.nc'
path = basedir+variable+'_ENSEMBLE/'+filename
# Get rid of ensmn ones
path = [p for p in glob.glob(path) if 'ensmn' not in p]

In [29]:
def preprocess(ds):
    nt = len(ds[time])
    # Recover ensemble details from path
    path = ds.encoding["source"]
    filename = path.split('/')[-1]
    filenamelist = filename.split('_')
    ensembleid = filenamelist[1]
    memberid = filenamelist[2]
    if memberid[0:3]=='neg':
        sign = -1
    else:
        sign=1
    member = int(memberid[-2:])*sign
    startyear = int(filenamelist[3])
    
    # Expand init dimension
    init = cftime.DatetimeNoLeap(startyear,1,1)
    ds = ds.expand_dims({'init':[init]})
    # Expand member dimension
    ds = ds.expand_dims({'member':[member]})
    # Assign "lead" coordinate
    ds = ds.rename({time:'lead'}).assign_coords({'lead':np.arange(1,1+nt)})
    ds['lead'].attrs['units']='months'
        
    ds = ds.chunk({'lead':-1})
    return ds

In [30]:
ds = xr.open_mfdataset(path,preprocess=preprocess)[filenamevariable].chunk({'init':3})
ds

<xarray.DataArray 'NPP' (member: 40, init: 6, lead: 120, YT_OCEAN: 200,
                         XT_OCEAN: 360)>
dask.array<rechunk-merge, shape=(40, 6, 120, 200, 360), dtype=float32, chunksize=(1, 3, 120, 200, 360), chunktype=numpy.ndarray>
Coordinates:
  * member    (member) int64 -20 -19 -18 -17 -16 -15 -14 ... 15 16 17 18 19 20
  * init      (init) object 0022-01-01 00:00:00 ... 0295-01-01 00:00:00
  * XT_OCEAN  (XT_OCEAN) float64 -279.5 -278.5 -277.5 -276.5 ... 77.5 78.5 79.5
  * YT_OCEAN  (YT_OCEAN) float64 -81.5 -80.5 -79.5 -78.5 ... 86.5 87.5 88.5 89.5
  * lead      (lead) int64 1 2 3 4 5 6 7 8 9 ... 113 114 115 116 117 118 119 120
Attributes:
    long_name:  JPROD_NLG_NONDIATOMS_100[D=1]+JPROD_NLG_DIATOMS_100[D=1]+JPRO...

In [31]:
if variable=='NPP':
    ds = ds.rename({'XT_OCEAN':'xt_ocean','YT_OCEAN':'yt_ocean'})

## Load control

In [32]:
controlnames = {'NPP':'NPP_100','SST':'sst','MLD':'mld'}

localdir = 'CONTROL/'+variable+'/'
filename = controlnames[variable]+'_0001_0300.nc'
path = basedir+localdir+filename

In [33]:
control = xr.open_mfdataset(path).rename({time:'time'})[filenamevariable]
if variable=='NPP':
    control = control.rename({'XT_OCEAN':'xt_ocean','YT_OCEAN':'yt_ocean'})

# Calculate PPP

In [34]:
ppp = esm4ppe.calc_ppp(ds,control,groupby=None,frequency='monthly',nlead=120)

/home/graemem/.conda/envs/climpred/lib/python3.11/site-packages/xarray/core/indexing.py:1446: PerformanceWarning: Slicing with an out-of-order index is generating 10 times more chunks
  return self.array[key]


In [35]:
ppp

<xarray.DataArray 'NPP' (lead: 120, yt_ocean: 200, xt_ocean: 360)>
dask.array<sub, shape=(120, 200, 360), dtype=float32, chunksize=(1, 200, 360), chunktype=numpy.ndarray>
Coordinates:
  * xt_ocean  (xt_ocean) float64 -279.5 -278.5 -277.5 -276.5 ... 77.5 78.5 79.5
  * yt_ocean  (yt_ocean) float64 -81.5 -80.5 -79.5 -78.5 ... 86.5 87.5 88.5 89.5
  * lead      (lead) int64 1 2 3 4 5 6 7 8 9 ... 113 114 115 116 117 118 119 120

# Load and save PPP

In [36]:
savedir = "/projects/SOCCOM/graemem/projects/esm4_ppe/data/processed/esm2m/"
localdir = 'verify.metric-ppp/'
filename = 'ocean.'+variable+'.monthly.nc'
savepath = savedir+localdir+filename

In [38]:
with ProgressBar():
    ppp = ppp.load()
ppp.to_netcdf(savepath)